# Advanced Embeddings

[Notebooks 03](/courses/llm-eng/03-rag-concepts.html) and [07](/courses/llm-eng/07-rag-pipeline.html) treat `text-embedding-3-small` as a black box: hand it text, receive a 1536-dimensional vector. This deep dive opens that box. We survey the embedding model landscape — the MTEB benchmark, bi-encoder vs. cross-encoder families, and how to read retrieval metrics for financial corpora — then work through four concrete techniques that a principal AI engineer must be able to reason about: (1) **Matryoshka Representation Learning**, which allows `text-embedding-3-*` vectors to be truncated without catastrophic quality loss, with implications for storage cost; (2) **contrastive fine-tuning**, the supervised objective that adapts a general-purpose encoder to domain-specific text using query–passage pairs with hard negatives; (3) **dimensionality reduction via PCA + UMAP**, which surfaces the geometric structure of SEC filing embeddings and reveals clustering by document section; and (4) **ColBERT late interaction**, a retrieval paradigm that operates on token-level vectors rather than a single document embedding and achieves higher precision at the cost of index size. All experiments run against the 20-chunk SEC filings corpus from notebook 07.

Setup:

In [ ]:
#| echo: false
import os, json
import numpy as np
from dotenv import load_dotenv
load_dotenv()

import openai
from pydantic import BaseModel
from typing import Optional, Type

PRICES = {
    "gpt-4o":      {"input": 2.50,  "output": 10.00},
    "gpt-4o-mini": {"input": 0.15,  "output": 0.60},
}

class LLMClient:
    def __init__(self, model="gpt-4o-mini", temperature=0.0):
        self.model = model; self.temperature = temperature
        self._client = openai.OpenAI()
        self._in = 0; self._out = 0

    def complete(self, messages, *, response_format=None):
        if response_format is not None:
            resp = self._client.beta.chat.completions.parse(
                model=self.model, messages=messages,
                temperature=self.temperature, response_format=response_format)
        else:
            resp = self._client.chat.completions.create(
                model=self.model, messages=messages, temperature=self.temperature)
        if resp.usage:
            self._in += resp.usage.prompt_tokens; self._out += resp.usage.completion_tokens
        if response_format is not None: return resp.choices[0].message.parsed
        return resp.choices[0].message.content

    @property
    def total_cost(self):
        if self.model not in PRICES: return 0.0
        p = PRICES[self.model]
        return (self._in * p["input"] + self._out * p["output"]) / 1_000_000

llm = LLMClient()

## Embedding Model Landscape

The [Massive Text Embedding Benchmark (MTEB)](https://huggingface.co/spaces/mteb/leaderboard) is the standard evaluation harness for embedding models. It covers 56 datasets across eight task types: retrieval, clustering, classification, reranking, summarization, semantic textual similarity, bitext mining, and pair classification. For RAG systems the most relevant task is **retrieval**, scored by **NDCG@10** (Normalized Discounted Cumulative Gain at rank 10), which penalizes models that place relevant documents in lower positions more than those that miss them entirely. **Clustering** scores — typically V-measure or NMI — tell us how well the embedding geometry separates topically distinct document sets, relevant when we want to understand whether section-level metadata filtering is necessary or whether the geometry handles it automatically.

<br>

**Bi-encoder vs. cross-encoder.** The embedding models we use in RAG are **bi-encoders**: they encode query and document independently into fixed-length vectors, and similarity is a simple dot product or cosine at retrieval time. This is efficient — documents can be pre-encoded offline, and a query lookup is a single forward pass plus an ANN search. **Cross-encoders** take the query and document concatenated and run full bidirectional self-attention over the pair, producing a single relevance score. Cross-encoders are dramatically more accurate but cannot pre-encode documents: every (query, document) pair requires a separate forward pass, so they are used only as rerankers over a small candidate set (as in notebook 07).

<br>

**Symmetric vs. asymmetric similarity.** **Symmetric** models are trained on sentence pairs where both items are of the same type (e.g., two news sentences). They work well for semantic textual similarity but poorly for retrieval, where a short query must be matched against a long passage. **Asymmetric** models are trained on (query, passage) pairs from human-annotated datasets like MS MARCO, natural questions, or — for domain-adapted models — proprietary corpora. `text-embedding-3-small`, the OpenAI `e5` family, and `BAAI/bge-*` are all asymmetric retrieval-oriented models.

<br>

The table below compares five models relevant to financial text retrieval. MTEB Retrieval scores are NDCG@10 averaged over the 15 BEIR datasets; Clustering scores are V-measure averaged over MTEB clustering datasets. The financial text adjustment is qualitative, reflecting model-specific evaluations on SEC/10-K-style corpora reported in open literature.

| Model | Dimensions | MTEB Retrieval (NDCG@10) | MTEB Clustering | Financial Text Notes | Cost / 1M tokens |
|:---|:---:|:---:|:---:|:---|:---:|
| `text-embedding-3-small` | 1536 (trunc.) | 55.4 | 49.0 | Strong general baseline; MRL truncation to 256 dims retains ~95% retrieval quality | $0.02 |
| `text-embedding-3-large` | 3072 (trunc.) | 64.6 | 56.0 | Best-in-class for financial entity disambiguation; 2× cost vs. small | $0.13 |
| `BAAI/bge-large-en-v1.5` | 1024 | 54.3 | 46.1 | Open weights; fine-tunable; good on regulatory dense text; requires self-hosting | free (self-hosted) |
| `intfloat/e5-large-v2` | 1024 | 56.0 | 44.6 | Consistent on financial NLP; needs `"query: "` / `"passage: "` prefix at inference | free (self-hosted) |
| `thenlper/gte-large` | 1024 | 52.2 | 46.2 | No special prefixes required; slightly weaker on long passages (>256 tokens) | free (self-hosted) |

**NOTE:** NDCG@10 on general BEIR benchmarks does not directly predict performance on your specific financial corpus. The right evaluation discipline is: (1) embed your actual corpus, (2) curate 50–100 representative queries with human-labeled relevant passages, (3) compute Recall@$k$ and MRR against that labeled set. We build exactly this in the benchmarking section at the end of this notebook.

:::{.callout-tip}
For a production fintech system, start with `text-embedding-3-small` at default 1536 dimensions. It offers the best balance of quality, cost, and zero deployment overhead (no GPU, no self-hosting). Move to `text-embedding-3-large` if retrieval quality on your labeled test set is insufficient; move to a fine-tuned open model only if you have $\geq 1{,}000$ labeled query–passage pairs and a reproducible training pipeline.

:::

## The SEC Filings Corpus

We reuse the 20-chunk corpus from [notebook 07](/courses/llm-eng/07-rag-pipeline.html) verbatim so that all experiments are directly comparable. The corpus is structured into four sections — `risk_factors`, `mda`, `capital_liquidity`, and `guidance` — each with five passages drawn from realistic 10-K language. We tag each chunk with its section so we can use section labels as ground-truth cluster assignments in the visualization section.

In [ ]:
CORPUS = [
    # Section: risk_factors (indices 0-4)
    {"text": "Interest rate risk represents one of the most significant market risks facing the firm. A 100 basis point increase in interest rates would reduce the fair value of our fixed-rate debt portfolio by approximately $2.3 billion.", "section": "risk_factors"},
    {"text": "Credit risk arises from the potential that a counterparty will fail to perform its obligations. We manage credit risk through diversification, collateral requirements, and credit limits by counterparty.", "section": "risk_factors"},
    {"text": "Operational risk includes the risk of loss resulting from inadequate or failed internal processes, people, systems, or external events, including cybersecurity threats and technology failures.", "section": "risk_factors"},
    {"text": "Our derivatives portfolio had a notional value of $1.2 trillion at year-end. Net market value exposure after netting and collateral was $18.4 billion, primarily concentrated in interest rate and foreign exchange derivatives.", "section": "risk_factors"},
    {"text": "Our Value at Risk (VaR) at the 99th percentile for a one-day holding period was $142 million, reflecting the diversified nature of our trading portfolios across equities, fixed income, and commodities.", "section": "risk_factors"},

    # Section: mda (indices 5-9)
    {"text": "Net revenues for the fiscal year were $47.4 billion, an increase of 8% compared to the prior year. The increase was driven primarily by higher net interest income reflecting the rising interest rate environment.", "section": "mda"},
    {"text": "Investment banking revenues decreased 23% to $6.1 billion, reflecting lower advisory fees amid reduced M&A activity and a challenging environment for equity and debt underwriting.", "section": "mda"},
    {"text": "Return on equity for the year was 12.4%, compared to 15.1% in the prior year. Book value per share increased to $312.50, up from $290.20.", "section": "mda"},
    {"text": "Net interest margin expanded 18 basis points to 2.94%, driven by higher short-term rates partially offset by increased funding costs. Provision for credit losses increased to $2.1 billion, reflecting normalization from historically low levels.", "section": "mda"},
    {"text": "Assets under management in our investment management segment grew to $2.8 trillion, an increase of 6% from prior year. Prime brokerage revenues increased 12% to $4.3 billion on higher client balances and margin loan activity.", "section": "mda"},

    # Section: capital_liquidity (indices 10-14)
    {"text": "Our Common Equity Tier 1 (CET1) capital ratio was 14.8% at year-end, well above the regulatory minimum of 4.5% and our internal target of 13%.", "section": "capital_liquidity"},
    {"text": "We maintain a liquidity coverage ratio (LCR) of 128%, exceeding the regulatory requirement of 100%. Our high-quality liquid assets totaled $280 billion at year-end.", "section": "capital_liquidity"},
    {"text": "Under Basel III framework requirements, our leverage ratio was 5.8%, comfortably above the 3% minimum. Our total risk-weighted assets were $1.47 trillion at year-end, consistent with prior year.", "section": "capital_liquidity"},
    {"text": "We are subject to FINRA Rule 4110 and SEC Rule 15c3-1 (the Net Capital Rule), which require us to maintain minimum net capital of not less than the greater of $250,000 or 2% of aggregate debit items. Our net capital exceeded the minimum by $12.4 billion.", "section": "capital_liquidity"},
    {"text": "The liquidity stress test results indicate the firm could withstand a 30-day severe market stress scenario while maintaining positive liquidity. The Internal Liquidity Adequacy Assessment Process (ILAAP) was completed in Q3 and reviewed by the Board Risk Committee.", "section": "capital_liquidity"},

    # Section: guidance (indices 15-19)
    {"text": "Looking ahead to fiscal 2025, management expects continued revenue growth in the range of 4-6%, supported by a stable rate environment and improving capital markets activity.", "section": "guidance"},
    {"text": "We plan to return $8 billion to shareholders through dividends and share repurchases in fiscal 2025, subject to regulatory approval and market conditions.", "section": "guidance"},
    {"text": "Earnings per share for fiscal 2024 were $42.30, compared to $47.20 in the prior year, reflecting lower net income partially offset by the reduction in diluted share count from ongoing repurchases.", "section": "guidance"},
    {"text": "Management has identified three strategic priorities for fiscal 2025: (1) expanding the wealth management client base to $500 billion in AUM, (2) growing transaction banking revenues by 15%, and (3) reducing the expense ratio below 65%.", "section": "guidance"},
    {"text": "We expect our CET1 ratio to remain in the 13.5-15.0% range through fiscal 2025, subject to regulatory stress test outcomes. Any excess capital above 14.5% will be returned to shareholders under our capital return policy.", "section": "guidance"},
]

TEXTS = [c["text"] for c in CORPUS]
SECTIONS = [c["section"] for c in CORPUS]
print(f"Corpus: {len(CORPUS)} chunks")
for s in ["risk_factors", "mda", "capital_liquidity", "guidance"]:
    print(f"  {s}: {sum(1 for c in CORPUS if c['section'] == s)} chunks")

We define a thin embedding helper that supports an optional `dimensions` parameter — the key API feature explored in the next section:

In [ ]:
def embed(
    texts: list[str],
    model: str = "text-embedding-3-small",
    dimensions: int | None = None,
) -> np.ndarray:
    """Return embedding matrix of shape (len(texts), d)."""
    client = openai.OpenAI()
    kwargs = {"input": texts, "model": model}
    if dimensions is not None:
        kwargs["dimensions"] = dimensions  # <1>
    resp = client.embeddings.create(**kwargs)
    return np.array([item.embedding for item in resp.data], dtype=np.float32)


def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    """Cosine similarity between two 1-D or batched vectors."""
    if a.ndim == 1 and b.ndim == 1:
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))
    # batched: (n, d) vs (d,)
    norms = np.linalg.norm(b) + 1e-9
    return (b / norms) @ a.T  # <2>


# Pre-compute full-dimension corpus embeddings for reuse
CORPUS_VECS = embed(TEXTS)  # (20, 1536)
print(f"Corpus embedding matrix: {CORPUS_VECS.shape}")

1. The `dimensions` parameter is only supported by `text-embedding-3-*` models. Passing it to older models raises an API error.
2. For batched retrieval we normalize `b` (the query vector) and compute its dot product with the full corpus matrix in one call — this is the same $\hat{\mathbf{M}} \hat{\mathbf{q}}$ computation from notebook 03.

## Matryoshka Representation Learning and the `dimensions` Parameter

**Matryoshka Representation Learning (MRL)** is a training technique introduced by [Kusupati et al. (2022)](https://arxiv.org/abs/2205.13147) that trains an embedding model to simultaneously optimize representations at multiple nested dimensionalities. The idea: rather than training only the full $d$-dimensional output, MRL adds auxiliary losses at each of a set of prefixes $\{m_1, m_2, \ldots, m_L\}$ where $m_1 < m_2 < \cdots < m_L = d.$ The total training objective is:

$$\mathcal{L}_{\text{MRL}} = \sum_{l=1}^{L} c_l \cdot \mathcal{L}\bigl(\mathbf{z}_{[1:m_l]}\bigr)$$

where $\mathbf{z}_{[1:m_l]}$ denotes the first $m_l$ coordinates of the embedding and $c_l > 0$ are importance weights. The practical consequence is that the [first $m$ dimensions]{.mark} of an MRL embedding are themselves a good $m$-dimensional embedding — you can truncate the vector after training and retain most of the retrieval quality, without any separate training run or dimensionality reduction step.

OpenAI's `text-embedding-3-small` (1536-dim) and `text-embedding-3-large` (3072-dim) are both trained with MRL. The API exposes this via the `dimensions` parameter, which truncates and re-normalizes the output server-side. This is significant for a production system:

| Dimension | Vector bytes (float32) | Relative storage | Note |
|:---:|:---:|:---:|:---|
| 1536 | 6,144 B | 100% | Default |
| 1024 | 4,096 B | 67% | Modest saving, minimal quality drop |
| 512 | 2,048 B | 33% | Good Pareto point for financial text |
| 256 | 1,024 B | 17% | Noticeable quality drop on hard queries |

We implement `EmbeddingDimensionStudy` to measure Recall@5 against dimension size on our 10-question labeled test set:

In [ ]:
# Labeled test set: (query, ground-truth chunk index)
EVAL_SET = [
    ("What is the CET1 capital ratio?", 10),
    ("What is the LCR and HQLA balance?", 11),
    ("How did investment banking revenues perform?", 6),
    ("What is the net interest margin?", 8),
    ("What are the cybersecurity and technology risks?", 2),
    ("What is the Basel III leverage ratio?", 12),
    ("What is the VaR at the 99th percentile?", 4),
    ("FINRA Rule 4110 Net Capital Rule compliance", 13),
    ("What is the earnings per share?", 17),
    ("What are AUM and prime brokerage revenues?", 9),
]


class EmbeddingDimensionStudy:
    """Measure Recall@k across embedding dimension sizes."""

    def __init__(self, corpus: list[dict], eval_set: list[tuple[str, int]]):
        self._corpus = corpus
        self._texts = [c["text"] for c in corpus]
        self._eval = eval_set

    def recall_at_k(
        self,
        dimensions: int,
        k: int = 5,
        model: str = "text-embedding-3-small",
    ) -> float:
        """Return Recall@k for the given embedding dimension."""
        doc_vecs = embed(self._texts, model=model, dimensions=dimensions)  # <1>
        doc_norms = np.linalg.norm(doc_vecs, axis=1, keepdims=True) + 1e-9
        doc_vecs_n = doc_vecs / doc_norms

        hits = 0
        for query, gt_idx in self._eval:
            q_vec = embed([query], model=model, dimensions=dimensions)[0]
            q_vec = q_vec / (np.linalg.norm(q_vec) + 1e-9)
            scores = doc_vecs_n @ q_vec  # (20,)
            top_k = set(np.argsort(scores)[::-1][:k])  # <2>
            if gt_idx in top_k:
                hits += 1

        return hits / len(self._eval)

    def sweep(
        self,
        dimensions: list[int],
        k: int = 5,
        model: str = "text-embedding-3-small",
    ) -> dict[int, float]:
        """Run recall measurement for each dimension size."""
        results = {}
        for d in dimensions:
            r = self.recall_at_k(d, k=k, model=model)
            results[d] = r
            print(f"  dim={d:4d}  Recall@{k}={r:.2f}")
        return results


study = EmbeddingDimensionStudy(CORPUS, EVAL_SET)
print("Sweeping dimension sizes for text-embedding-3-small:")
sweep_results = study.sweep([256, 512, 1024, 1536], k=5)

1. Both the corpus and each query must be embedded at the same `dimensions` setting — mixing dimension sizes produces incomparable vectors since the truncation removes different coordinate subsets.
2. We use a set for $O(1)$ membership testing; the ground-truth index is either in the top-$k$ or not.

We plot the Pareto frontier — Recall@5 vs. storage cost — to show the diminishing returns of larger dimensions:

In [ ]:
#| code-fold: true
import matplotlib.pyplot as plt

dims = sorted(sweep_results.keys())
recalls = [sweep_results[d] for d in dims]
# Storage cost in bytes for 1M vectors at float32
storage_gb = [d * 4 * 1_000_000 / 1e9 for d in dims]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Left: Recall vs dimension
ax = axes[0]
ax.plot(dims, recalls, marker="o", linewidth=2, color="#2166ac", markersize=7)
ax.axhline(recalls[-1], linestyle="--", alpha=0.4, color="gray", label="1536-dim baseline")
for d, r in zip(dims, recalls):
    ax.annotate(f"{r:.2f}", (d, r), textcoords="offset points", xytext=(0, 8),
                ha="center", fontsize=9)
ax.set_xlabel("Embedding dimensions")
ax.set_ylabel("Recall@5")
ax.set_title("Recall@5 vs. dimension (MRL truncation)")
ax.set_ylim(0, 1.15)
ax.legend(fontsize=9)
ax.grid(linestyle="dotted", alpha=0.6)

# Right: Recall vs storage
ax2 = axes[1]
ax2.plot(storage_gb, recalls, marker="s", linewidth=2, color="#d6604d", markersize=7)
for d, gb, r in zip(dims, storage_gb, recalls):
    ax2.annotate(f"dim={d}", (gb, r), textcoords="offset points", xytext=(4, 4),
                 fontsize=8)
ax2.set_xlabel("Storage for 1M vectors (GB, float32)")
ax2.set_ylabel("Recall@5")
ax2.set_title("Pareto frontier: quality vs. storage")
ax2.grid(linestyle="dotted", alpha=0.6)

plt.tight_layout()
plt.show()

**Figure.** Left: Recall@5 as a function of the `dimensions` parameter. The MRL training objective means the first 512 coordinates already capture most of the retrieval signal. Right: the storage Pareto frontier — moving from 1536 to 512 dimensions reduces vector storage by 67% with negligible quality loss on this corpus, a compelling trade-off for large-scale deployments.

:::{.callout-note}
On small corpora (20 chunks) the Recall@5 ceiling is easily saturated at all dimension sizes. The dimension effect becomes more pronounced at scale — 100k+ chunks — where the ANN index must separate many geometrically similar vectors. When calibrating on your own corpus, always evaluate on a corpus that is representative of production size.

:::

## Fine-Tuning Embeddings for Domain Adaptation

General-purpose embedding models are trained on broad web corpora. Financial text uses specialized vocabulary — CUSIP, CET1, LCR, FINRA Rule 4110, ILAAP — that is either absent from training data or treated as low-frequency noise. **Contrastive fine-tuning** adapts a bi-encoder to this domain by training on (query, positive passage, hard negatives) triples curated from the target corpus.

The training objective is the **InfoNCE** (in-batch contrastive) loss. Given a query $q$, a positive passage $p^+$, and $N$ in-batch negatives $\{p^-_i\}$:

$$\mathcal{L} = -\log \frac{\exp(\text{sim}(q, p^+) / \tau)}{\exp(\text{sim}(q, p^+) / \tau) + \sum_{i=1}^{N} \exp(\text{sim}(q, p^-_i) / \tau)}$$

where $\text{sim}(\mathbf{a}, \mathbf{b}) = \mathbf{a}^\top \mathbf{b} / (\|\mathbf{a}\| \|\mathbf{b}\|)$ is cosine similarity and $\tau > 0$ is a temperature hyperparameter that controls the sharpness of the distribution. Lower $\tau$ makes the loss harder — the model must assign a much higher similarity to the positive than to any negative.

**Hard negatives** are critical for model quality. A random negative is a document that has nothing to do with the query — trivially distinguishable. A **hard negative** is a document that is topically related but not the correct answer. For our SEC corpus, a hard negative for the query "What is the CET1 capital ratio?" might be the LCR passage (also capital section, similar vocabulary) rather than the investment banking revenues passage. We mine hard negatives using BM25 top-$k$: retrieve the top-20 BM25 results and discard the ground-truth passage — the remaining passages are strong hard negatives because they share keywords with the query but are not the answer.

We implement `FineTuningDatasetBuilder` that generates training triples from the labeled evaluation set via BM25 hard negative mining:

In [ ]:
from rank_bm25 import BM25Okapi


class FineTuningDatasetBuilder:
    """Build contrastive training triples from a labeled query-passage set."""

    def __init__(self, corpus: list[dict]):
        self._corpus = corpus
        self._texts = [c["text"] for c in corpus]
        tokenized = [t.lower().split() for t in self._texts]
        self._bm25 = BM25Okapi(tokenized)  # <1>

    def _hard_negatives(
        self,
        query: str,
        gt_index: int,
        n_neg: int = 3,
        candidate_k: int = 15,
    ) -> list[str]:
        """Return top BM25 results excluding the ground-truth passage."""
        tokens = query.lower().split()
        scores = self._bm25.get_scores(tokens)
        ranked = np.argsort(scores)[::-1]
        negatives = [
            self._texts[i]
            for i in ranked[:candidate_k]
            if i != gt_index  # <2>
        ][:n_neg]
        return negatives

    def build(
        self,
        eval_set: list[tuple[str, int]],
        n_neg: int = 3,
    ) -> list[dict]:
        """Return training triples: {query, positive, negatives}."""
        triples = []
        for query, gt_idx in eval_set:
            positive = self._texts[gt_idx]
            negatives = self._hard_negatives(query, gt_idx, n_neg=n_neg)  # <3>
            triples.append({
                "query": query,
                "positive": positive,
                "negatives": negatives,
            })
        return triples


builder = FineTuningDatasetBuilder(CORPUS)
triples = builder.build(EVAL_SET, n_neg=3)

# Inspect one triple
t = triples[0]
print(f"Query:    {t['query']}")
print(f"Positive: {t['positive'][:80]}...")
print(f"\nHard negatives ({len(t['negatives'])})")
for i, neg in enumerate(t["negatives"]):
    print(f"  [{i+1}] {neg[:80]}...")

1. BM25 is used for hard negative mining because it retrieves lexically similar passages — exactly the kind of near-miss that a fine-tuned model must learn to discriminate.
2. We exclude the ground-truth index by direct comparison; in a real corpus with duplicate or paraphrase passages, exclude by chunk ID rather than text equality.
3. In practice, `n_neg = 7` to `15` negatives per query is common. More negatives per batch increases gradient signal but also GPU memory usage. With the `sentence-transformers` library, `MultipleNegativesRankingLoss` handles in-batch negatives automatically, so you only need to provide (query, positive) pairs.

We verify that the BM25 hard negatives are indeed topically related to the queries — the key property that makes them useful as training signal:

In [ ]:
print("Hard negative section distribution across all triples:")
from collections import Counter

# Map text → section for analysis
text_to_section = {c["text"]: c["section"] for c in CORPUS}

neg_sections = Counter()
pos_sections = Counter()
for t in triples:
    pos_sections[text_to_section[t["positive"]]] += 1
    for neg in t["negatives"]:
        neg_sections[text_to_section[neg]] += 1

print(f"  Positive sections: {dict(pos_sections)}")
print(f"  Negative sections: {dict(neg_sections)}")

**When is fine-tuning worth the effort?** The honest answer is: rarely for the average deployment. Fine-tuning requires (1) $\geq 500$ high-quality labeled (query, passage) pairs, (2) a reproducible training pipeline, (3) model serving infrastructure to host the custom weights, and (4) an eval harness that can detect regression on future prompt or corpus changes. All of this has ongoing maintenance cost. The practical decision rule: if `text-embedding-3-large` fails to meet your Recall@5 target on your labeled test set, fine-tune a self-hosted model like `bge-large-en-v1.5`; otherwise, use the API model and spend the engineering budget on hard negative mining for BM25 and metadata filtering.

## Dimensionality Reduction for Visualization

A 1536-dimensional vector is not human-inspectable. We reduce it to 2D using a two-stage pipeline: first **PCA** (Principal Component Analysis) to 50 dimensions, which removes noise while preserving most variance cheaply; then **UMAP** (Uniform Manifold Approximation and Projection) from 50 to 2 dimensions, which preserves local neighborhood structure better than PCA alone at low dimensionality.

The reason for the two-stage approach: UMAP is $O(n^2)$ in its naive form and benefits from a reduced input space. PCA is $O(n d^2)$ but fast in practice because `sklearn`'s implementation uses randomized SVD. The combination is the standard recipe in the embedding visualization literature — it is used in the original UMAP paper, the Sentence-BERT paper, and every embedding visualization we have seen in industry practice.

We implement `EmbeddingVisualizer` that wraps this pipeline and produces a color-coded scatter plot by document section:

In [ ]:
from sklearn.decomposition import PCA
import umap


class EmbeddingVisualizer:
    """PCA → UMAP dimensionality reduction for embedding visualization."""

    def __init__(self, pca_components: int = 50, random_state: int = 42):
        self._pca = PCA(n_components=pca_components, random_state=random_state)  # <1>
        self._umap = umap.UMAP(
            n_components=2,
            n_neighbors=5,  # <2>
            min_dist=0.1,
            metric="cosine",
            random_state=random_state,
        )

    def fit_transform(self, vectors: np.ndarray) -> np.ndarray:
        """Return 2D coordinates for each input vector."""
        reduced_pca = self._pca.fit_transform(vectors)   # <3>
        reduced_2d = self._umap.fit_transform(reduced_pca)  # <4>
        return reduced_2d


visualizer = EmbeddingVisualizer(pca_components=min(50, CORPUS_VECS.shape[0] - 1))
coords_2d = visualizer.fit_transform(CORPUS_VECS)
print(f"2D coordinates shape: {coords_2d.shape}")

1. We cap PCA components at `n_samples - 1` to avoid the rank constraint issue on small corpora; in production with $n \gg 50$ this is never binding.
2. `n_neighbors=5` is deliberately small because our corpus has only 20 points. For a real corpus of 10k+ chunks, use `n_neighbors=15` to `30`.
3. PCA is applied first: it is a global linear projection that efficiently removes the long tail of near-zero variance dimensions from a 1536-dimensional vector.
4. UMAP then finds a non-linear 2D manifold that preserves local neighborhoods in the 50-dimensional PCA space.

Plotting the 2D projection, colored by section:

In [ ]:
#| code-fold: true
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

SECTION_COLORS = {
    "risk_factors":     "#d6604d",
    "mda":              "#4393c3",
    "capital_liquidity": "#4dac26",
    "guidance":         "#b35900",
}
SECTION_LABELS = {
    "risk_factors": "Risk Factors",
    "mda": "MD&A",
    "capital_liquidity": "Capital & Liquidity",
    "guidance": "Guidance",
}

fig, ax = plt.subplots(figsize=(8, 6))

for i, (x, y) in enumerate(coords_2d):
    section = SECTIONS[i]
    color = SECTION_COLORS[section]
    ax.scatter(x, y, color=color, s=80, alpha=0.85, zorder=3, edgecolors="white", linewidths=0.5)
    ax.annotate(str(i), (x, y), fontsize=7, ha="center", va="center", color="white",
                fontweight="bold", zorder=4)

patches = [
    mpatches.Patch(color=SECTION_COLORS[s], label=SECTION_LABELS[s])
    for s in SECTION_COLORS
]
ax.legend(handles=patches, loc="best", fontsize=9, framealpha=0.9)
ax.set_title("PCA (50D) → UMAP (2D) projection of 20-chunk SEC corpus", fontsize=11)
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
ax.grid(linestyle="dotted", alpha=0.5)
plt.tight_layout()
plt.show()

**Figure.** PCA + UMAP projection of the 20-chunk SEC corpus, with each point labeled by its chunk index (0–19) and colored by document section. Chunks within the same section tend to cluster together, confirming that the embedding geometry encodes document section as a latent structure. Note that the `capital_liquidity` and `risk_factors` sections (both deal with regulatory and financial ratios) are closer in the UMAP space than either is to `mda` — consistent with the semantic proximity of those topics in real filing language.

## ColBERT Late Interaction

The three retrieval paradigms differ in where query–document interaction happens:

- **Bi-encoder** (our default): query and document are encoded independently into single vectors $\mathbf{q}, \mathbf{d} \in \mathbb{R}^d.$ Interaction is a single dot product $\mathbf{q}^\top \mathbf{d}.$ Fast to index (one vector per document), but all query–document reasoning is compressed into one number.
- **Cross-encoder**: query and document are concatenated and run through a transformer jointly. Full self-attention captures fine-grained interactions. Accurate but $O(n)$ latency per query — unusable for first-stage retrieval.
- **Late interaction (ColBERT)**: query and document are encoded separately into *token-level* vector sequences $\mathbf{E}_q \in \mathbb{R}^{|q| \times d}$ and $\mathbf{E}_d \in \mathbb{R}^{|d| \times d}.$ The relevance score is the **MaxSim** operator — for each query token, find its most similar document token and sum:

$$\text{score}(q, d) = \sum_{i \in q} \max_{j \in d} \mathbf{E}_q[i] \cdot \mathbf{E}_d[j]^\top$$

This is more expressive than a single vector dot product because each query token independently retrieves its best matching document token — a query about "CET1 ratio" will match the document token "CET1" strongly and the token "ratio" separately, capturing both. Document token matrices can be pre-computed and stored (though at 32× the storage of a bi-encoder). Query token matrices are computed at query time with a fast forward pass.

We implement a simplified `ColBERTRetriever` using `sentence-transformers` to produce token embeddings (we do not use the full ColBERT model or its specialized index, but the MaxSim computation is identical):

In [ ]:
import torch
from sentence_transformers import SentenceTransformer


class ColBERTRetriever:
    """Simplified ColBERT-style late interaction retriever using MaxSim."""

    def __init__(self, model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
        self._model = SentenceTransformer(model_name)
        self._tokenizer = self._model.tokenizer
        self._doc_token_vecs: list[np.ndarray] = []
        self._texts: list[str] = []

    def _token_embeddings(self, text: str) -> np.ndarray:
        """Return per-token embeddings of shape (seq_len, d)."""
        features = self._model.tokenize([text])
        with torch.no_grad():
            out = self._model.forward(features)  # <1>
        # token_embeddings: (1, seq_len, d)
        token_vecs = out["token_embeddings"][0].cpu().numpy()  # (seq_len, d)
        # L2-normalize each token vector
        norms = np.linalg.norm(token_vecs, axis=1, keepdims=True) + 1e-9
        return token_vecs / norms

    def index(self, corpus: list[dict]) -> None:
        """Pre-compute and store token embeddings for each document."""
        self._texts = [c["text"] for c in corpus]
        for text in self._texts:
            self._doc_token_vecs.append(self._token_embeddings(text))  # <2>
        print(f"Indexed {len(self._texts)} documents.")
        print(f"  Example doc token matrix shape: {self._doc_token_vecs[0].shape}")

    def _maxsim(self, q_vecs: np.ndarray, d_vecs: np.ndarray) -> float:
        """MaxSim score: sum over query tokens of their max similarity to doc tokens."""
        # q_vecs: (|q|, d), d_vecs: (|d|, d)
        sim_matrix = q_vecs @ d_vecs.T  # (|q|, |d|)  # <3>
        return float(sim_matrix.max(axis=1).sum())

    def retrieve(self, query: str, k: int = 5) -> list[dict]:
        """Retrieve top-k documents by MaxSim score."""
        q_vecs = self._token_embeddings(query)  # (|q|, d)
        scores = [self._maxsim(q_vecs, d_vecs) for d_vecs in self._doc_token_vecs]
        ranked = np.argsort(scores)[::-1][:k]
        return [
            {"text": self._texts[i], "score": scores[i], "index": int(i)}
            for i in ranked
        ]


colbert = ColBERTRetriever()
colbert.index(CORPUS)

1. We call `self._model.forward(features)` directly (bypassing the sentence-level pooling) to access the raw per-token output from the transformer encoder. The `out["token_embeddings"]` key is set by `sentence-transformers`' feature extraction pipeline.
2. For a 20-document corpus each document token matrix fits easily in memory. At scale, ColBERT's PLAID index compresses these token matrices using residual quantization — storing $\sim$32 bytes per token rather than $d \times 4$ bytes.
3. The $|q| \times |d|$ similarity matrix captures every pairwise token interaction. `max(axis=1)` picks the best document token for each query token; summing these maxima gives the ColBERT relevance score.

We compare ColBERT vs. bi-encoder retrieval on five financial queries with known ground-truth:

In [ ]:
COMPARISON_QUERIES = [
    ("What is the CET1 capital ratio?", 10),
    ("FINRA Rule 4110 Net Capital Rule compliance", 13),
    ("VaR 99th percentile one-day holding period", 4),
    ("Investment banking revenue change year over year", 6),
    ("Basel III leverage ratio minimum requirement", 12),
]

# Bi-encoder: use full CORPUS_VECS
CORPUS_VECS_N = CORPUS_VECS / (np.linalg.norm(CORPUS_VECS, axis=1, keepdims=True) + 1e-9)

def bi_encoder_retrieve(query: str, k: int = 5) -> list[dict]:
    q_vec = embed([query])[0]
    q_vec = q_vec / (np.linalg.norm(q_vec) + 1e-9)
    scores = CORPUS_VECS_N @ q_vec
    ranked = np.argsort(scores)[::-1][:k]
    return [{"text": TEXTS[i], "score": float(scores[i]), "index": int(i)} for i in ranked]


print(f"{'Query':<45} {'Bi-enc rank':>11} {'ColBERT rank':>13}")
print("-" * 72)
for query, gt_idx in COMPARISON_QUERIES:
    bi_results  = bi_encoder_retrieve(query, k=5)
    col_results = colbert.retrieve(query, k=5)

    bi_rank  = next((r+1 for r, res in enumerate(bi_results)  if res["index"] == gt_idx), ">5")
    col_rank = next((r+1 for r, res in enumerate(col_results) if res["index"] == gt_idx), ">5")

    print(f"{query[:44]:<45} {str(bi_rank):>11} {str(col_rank):>13}")

The FINRA Rule 4110 query illustrates the key advantage of late interaction: the query token "4110" must match the document token "4110" exactly. A bi-encoder compresses this specificity into a 1536-dimensional average; the MaxSim operator evaluates it token by token, so an exact identifier match contributes its full cosine score rather than being diluted by the rest of the passage.

## Embedding Storage and Indexing

The choice of ANN index depends on three variables: corpus size $n$, query rate (QPS), and acceptable recall tradeoff. Three index families dominate production deployments:

**Flat index (exact).** Brute-force $O(n \cdot d)$ cosine search. Recall = 100% by definition. Appropriate for $n < 50{,}000$ when latency is not a constraint, or as a ground-truth baseline for benchmarking ANN quality. ChromaDB uses flat search for small collections before automatically switching to HNSW.

**HNSW (Hierarchical Navigable Small World).** A graph-based ANN index that builds a multi-layer navigable small world graph over the vector set. Query complexity is approximately $O(\log n)$ at the cost of a one-time build and increased memory usage. Two key parameters: (1) `M` — the number of bidirectional edges per node at each layer; higher `M` means better recall and faster queries at the cost of more memory and slower build. Typical values: `M=16` (default) to `M=64` (high-recall). (2) `ef_construction` — the size of the dynamic candidate list during index construction; higher values produce better graph quality at the cost of build time. Typical values: `ef_construction=100` to `ef_construction=400`. HNSW is the right choice for $10^4 \leq n \leq 10^7$ at query rates of 10–10,000 QPS.

**IVF (Inverted File Index).** Clusters the vector space into $\sqrt{n}$ Voronoi cells using $k$-means. At query time, only the $n_{\text{probe}}$ nearest cells are searched. IVF-PQ (with product quantization for vector compression) is the right choice for $n > 10^7$ where HNSW memory becomes prohibitive: IVF-PQ can represent 100M vectors in RAM while HNSW cannot.

ChromaDB uses HNSW by default via [hnswlib](https://github.com/nmslib/hnswlib). We can inspect and adjust its settings at collection creation time:

In [ ]:
import chromadb
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

client = chromadb.EphemeralClient()  # <1>

embed_fn = OpenAIEmbeddingFunction(
    api_key=os.environ["OPENAI_API_KEY"],
    model_name="text-embedding-3-small",
)

# Default HNSW settings
col_default = client.create_collection(
    name="default-hnsw",
    embedding_function=embed_fn,
    metadata={
        "hnsw:space": "cosine",    # <2>
        "hnsw:M": 16,              # <3>
        "hnsw:construction_ef": 100,  # <4>
        "hnsw:search_ef": 10,      # <5>
    }
)

# High-recall HNSW for compliance queries (quality over speed)
col_highrecall = client.create_collection(
    name="highrecall-hnsw",
    embedding_function=embed_fn,
    metadata={
        "hnsw:space": "cosine",
        "hnsw:M": 32,
        "hnsw:construction_ef": 200,
        "hnsw:search_ef": 100,     # <6>
    }
)

print("Default collection metadata:", col_default.metadata)
print("High-recall collection metadata:", col_highrecall.metadata)

1. `EphemeralClient` stores the index in memory only — no disk writes. Equivalent to `PersistentClient` for API exploration but simpler to clean up.
2. `hnsw:space` sets the distance metric for the HNSW graph. Use `"cosine"` for normalized text embeddings; `"l2"` is appropriate for image embeddings or other non-normalized vectors.
3. `hnsw:M` controls graph connectivity. Each new node connects to `M` neighbors at each layer. Doubling `M` roughly doubles memory usage but improves recall significantly for `M < 32.`
4. `hnsw:construction_ef` is the size of the dynamic candidate list during index build. Higher values produce a better-connected graph but slow down indexing. Set once and do not change after documents are added.
5. `hnsw:search_ef` controls the beam width during query time. Higher values trade query latency for higher recall. This can be changed per-query; ChromaDB uses `ef = max(n_results, search_ef)` internally.
6. For a compliance-critical RAG system where missing a relevant document has regulatory consequences, `search_ef=100` or higher is appropriate. For a lower-stakes financial chatbot, the default `search_ef=10` is fine.

:::{.callout-caution}
HNSW parameters cannot be changed after documents have been indexed. Always configure `M` and `construction_ef` before adding documents. If you need to change them, you must delete the collection, recreate it with the new settings, and re-index — which on a large corpus means re-running the embedding API calls at full cost.

:::

## Benchmarking All Approaches

We assemble `EmbeddingBenchmark` to measure Recall@5, MRR (Mean Reciprocal Rank), and p50/p99 query latency for three retrieval configurations on our 10-question labeled test set. **MRR** is the mean of $1/\text{rank}$ where rank is the position of the ground-truth chunk in the retrieved list; it penalizes a ground-truth chunk at rank 5 more gently than a ranking metric like NDCG but more gently than Recall@1. We add API cost per 1M tokens as a column since cost is a first-class constraint in production systems.

In [ ]:
import time


class EmbeddingBenchmark:
    """Measure Recall@k, MRR, and latency for multiple retrieval methods."""

    def __init__(self, eval_set: list[tuple[str, int]]):
        self._eval = eval_set

    def _mrr(self, retrieve_fn, k: int = 5) -> tuple[float, float, float, float]:
        """Return (Recall@k, MRR, p50_ms, p99_ms)."""
        recall_hits = 0
        reciprocal_ranks = []
        latencies_ms = []

        for query, gt_idx in self._eval:
            t0 = time.perf_counter()
            results = retrieve_fn(query, k)
            latencies_ms.append((time.perf_counter() - t0) * 1000)

            indices = [r["index"] for r in results]  # <1>
            if gt_idx in indices[:k]:
                recall_hits += 1

            rank = next((i + 1 for i, idx in enumerate(indices) if idx == gt_idx), None)
            reciprocal_ranks.append(1.0 / rank if rank else 0.0)  # <2>

        n = len(self._eval)
        lats = sorted(latencies_ms)
        return (
            recall_hits / n,
            float(np.mean(reciprocal_ranks)),
            lats[int(0.50 * len(lats))],
            lats[int(0.99 * len(lats))],
        )

    def run(
        self,
        methods: dict,  # name → (retrieve_fn, cost_per_1m_tokens)
        k: int = 5,
    ) -> dict:
        results = {}
        for name, (fn, cost) in methods.items():
            recall, mrr, p50, p99 = self._mrr(fn, k=k)
            results[name] = {"recall": recall, "mrr": mrr, "p50_ms": p50, "p99_ms": p99, "cost": cost}
        return results


def bi_full_retrieve(query: str, k: int) -> list[dict]:
    q_vec = embed([query])[0]
    q_vec = q_vec / (np.linalg.norm(q_vec) + 1e-9)
    scores = CORPUS_VECS_N @ q_vec
    ranked = np.argsort(scores)[::-1][:k]
    return [{"text": TEXTS[i], "score": float(scores[i]), "index": int(i)} for i in ranked]


def bi_256_retrieve(query: str, k: int) -> list[dict]:  # <3>
    doc_vecs = embed(TEXTS, dimensions=256)
    doc_vecs = doc_vecs / (np.linalg.norm(doc_vecs, axis=1, keepdims=True) + 1e-9)
    q_vec = embed([query], dimensions=256)[0]
    q_vec = q_vec / (np.linalg.norm(q_vec) + 1e-9)
    scores = doc_vecs @ q_vec
    ranked = np.argsort(scores)[::-1][:k]
    return [{"text": TEXTS[i], "score": float(scores[i]), "index": int(i)} for i in ranked]


def colbert_retrieve(query: str, k: int) -> list[dict]:
    return colbert.retrieve(query, k=k)


benchmark = EmbeddingBenchmark(EVAL_SET)

# (retrieve_fn, cost_per_1M_tokens_USD)
METHODS = {
    "bi-encoder 1536d":  (bi_full_retrieve, 0.02),
    "bi-encoder 256d":   (bi_256_retrieve,  0.02),   # same API price, less storage
    "ColBERT (MiniLM)": (colbert_retrieve, 0.00),    # self-hosted
}

print("Running benchmark...")
bench_results = benchmark.run(METHODS, k=5)

1. We standardize on `index` (corpus position) as the identifier for all retrieval methods. The ColBERT and bi-encoder implementations both return `{"index": int, ...}` so the benchmark code is method-agnostic.
2. MRR is 0 if the ground truth does not appear anywhere in the top-$k$ results — a missed query contributes 0 to the mean, so MRR is sensitive to failures in a way that Recall@5 is not.
3. We re-embed the corpus on each call for a fair latency comparison. In production the 256-dim corpus embeddings would be pre-computed, making the latency advantage of 256d over 1536d only the ANN search time difference (modest), not the embedding time difference (significant).

Displaying the benchmark results table:

In [ ]:
#| code-fold: true
print(f"{'Method':<22} {'Recall@5':>9} {'MRR':>7} {'p50 (ms)':>10} {'p99 (ms)':>10} {'Cost/1M tok':>12}")
print("-" * 74)
for name, r in bench_results.items():
    cost_str = f"${r['cost']:.2f}" if r["cost"] > 0 else "free"
    print(
        f"{name:<22} {r['recall']:>9.2f} {r['mrr']:>7.3f}"
        f" {r['p50_ms']:>10.1f} {r['p99_ms']:>10.1f} {cost_str:>12}"
    )

The benchmark table exposes a few practical takeaways. The 256-dim bi-encoder trades 83% storage cost reduction for a small MRR degradation — the Pareto point for storage-constrained deployments. ColBERT's MiniLM-based implementation gains on exact-identifier queries (FINRA Rule 4110, VaR percentile) relative to the bi-encoder, at the cost of higher per-query latency due to the per-token forward pass and the $|q| \times |d|$ similarity matrix computation.

:::{.callout-note}
The p99 latency for all methods is dominated by the OpenAI embedding API call, not the retrieval computation. In a production system, pre-computing corpus embeddings offline eliminates this cost from the query path — the query path then incurs only the cost of embedding the user query (one API call) plus the ANN lookup (sub-millisecond for HNSW at our corpus size).

:::

## Appendix: Contrastive Loss Derivation

The InfoNCE loss from the fine-tuning section has a clean probabilistic interpretation. Define the conditional probability that $p^+$ is the relevant passage for query $q$ given the candidate set $\{p^+\} \cup \{p^-_i\}_{i=1}^N$:

$$P(p^+ \mid q) = \frac{\exp(\text{sim}(q, p^+) / \tau)}{\exp(\text{sim}(q, p^+) / \tau) + \sum_{i=1}^{N} \exp(\text{sim}(q, p^-_i) / \tau)}$$

This is a softmax over the candidate set with temperature $\tau.$ The InfoNCE objective is simply the negative log-likelihood:

$$\mathcal{L} = -\log P(p^+ \mid q) = -\frac{\text{sim}(q, p^+)}{\tau} + \log \left[ \exp\left(\frac{\text{sim}(q, p^+)}{\tau}\right) + \sum_{i=1}^N \exp\left(\frac{\text{sim}(q, p^-_i)}{\tau}\right) \right]$$

Minimizing $\mathcal{L}$ is equivalent to maximizing $P(p^+ \mid q)$ — that is, assigning the highest softmax probability to the positive passage. The temperature $\tau$ controls how sharply the model must separate positives from negatives: at $\tau \to \infty$ the softmax is uniform (no gradient), at $\tau \to 0$ the loss degenerates to a winner-takes-all hard margin. Typical values in practice are $\tau \in [0.01, 0.1].$

**In-batch negatives.** In practice we do not construct explicit negative passages: within a training batch of $B$ (query, positive) pairs, each query treats all other $B - 1$ positives as its negatives. This is called **in-batch negative sampling**. The batch then contributes $B \times (B-1)$ implicit negative pairs per step, making each gradient update far more informative than a single (query, positive, single-negative) triple. This is the strategy used by `sentence-transformers`' `MultipleNegativesRankingLoss`.

## Appendix: HNSW Parameter Reference

Quick reference table for tuning HNSW in ChromaDB / hnswlib for financial retrieval workloads.

| Parameter | Default | Low (speed) | High (recall) | When to increase |
|:---|:---:|:---:|:---:|:---|
| `hnsw:M` | 16 | 8 | 64 | Corpus with many near-duplicate passages (e.g., boilerplate regulatory language across filings) |
| `hnsw:construction_ef` | 100 | 50 | 400 | When offline index build time is cheap and retrieval quality is critical |
| `hnsw:search_ef` | 10 | 10 | 200 | Compliance-critical queries; tune via Recall@k evaluation; set per-collection |
| `hnsw:space` | `"l2"` | — | — | Always set to `"cosine"` for text embeddings; `"l2"` is default but wrong for normalized vectors |

: HNSW parameter guide for ChromaDB on SEC filing corpora. {tbl-colwidths="[18,10,12,12,48]"}

**NOTE:** Increasing `M` from 16 to 32 roughly doubles memory usage. At 1M vectors of dimension 1536 (float32), the vector matrix alone is 6 GB; the HNSW graph overlay at `M=16` adds approximately 2 GB. At `M=64` the graph overhead approaches the vector matrix size — a significant cost at the 10M-vector scale typical of large financial institutions ingesting filings across thousands of registrants.

---

$\blacksquare$